In [3]:
import numpy as np
import pandas as pd
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sklearn.decomposition import TruncatedSVD
from sklearn.manifold import TSNE

from bokeh.io import curdoc, push_notebook, output_notebook
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.plotting import figure, show

from ipywidgets import interact

import warnings
warnings.filterwarnings("ignore")

In [4]:
data = pd.read_csv('skincare_products_clean.csv')
data

,product_name,product_url,product_type,clean_ingreds,price
0,The Ordinary Natural Moisturising Factors + HA...,https://www.lookfantastic.com/the-ordinary-nat...,Moisturiser,"['capric triglyceride', 'cetyl alcohol', 'prop...",£5.20
1,CeraVe Facial Moisturising Lotion SPF 25 52ml,https://www.lookfantastic.com/cerave-facial-mo...,Moisturiser,"['homosalate', 'glycerin', 'octocrylene', 'eth...",£13.00
2,The Ordinary Hyaluronic Acid 2% + B5 Hydration...,https://www.lookfantastic.com/the-ordinary-hya...,Moisturiser,"['sodium hyaluronate', 'sodium hyaluronate', '...",£6.20
3,AMELIORATE Transforming Body Lotion 200ml,https://www.lookfantastic.com/ameliorate-trans...,Moisturiser,"['ammonium lactate', 'c12-15', 'glycerin', 'pr...",£22.50
4,CeraVe Moisturising Cream 454g,https://www.lookfantastic.com/cerave-moisturis...,Moisturiser,"['glycerin', 'cetearyl alcohol', 'capric trigl...",£16.00
...,...,...,...,...,...
1133,Elemis Life Elixirs Embrace Bath and Shower El...,https://www.lookfantastic.com/elemis-life-elix...,Bath Oil,"['prunus amygdalus dulcis', 'tipa-laureth sulf...",£55.00
1134,Love Boo Splendidly Soothing Bath Soak (250ml),https://www.lookfantastic.com/love-boo-splendi...,Bath Oil,"['sodium lauroyl', 'sodium cocoamphoacetate', ...",£10.99
1135,Elemis Life Elixirs Fortitude Bath and Shower ...,https://www.lookfantastic.com/elemis-life-elix...,Bath Oil,"['prunus amygdalus dulcis', 'tipa-laureth sulf...",£55.00
1136,Connock London Kukui Oil Soothing Bath & Showe...,https://www.lookfantastic.com/connock-london-k...,Bath Oil,"['capric triglyceride', 'peg-40 sorbitan perol...",£36.00


In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1138 entries, 0 to 1137
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   product_name   1138 non-null   object
 1   product_url    1138 non-null   object
 2   product_type   1138 non-null   object
 3   clean_ingreds  1138 non-null   object
 4   price          1138 non-null   object
dtypes: object(5)
memory usage: 44.6+ KB


In [6]:
print(data.columns)

Index(['product_name', 'product_url', 'product_type', 'clean_ingreds',
       'price'],
      dtype='object')


In [7]:
def clean_ingredients(text):

    text = str(text).lower()

    text = re.sub(r'[\[\]\(\)\']', '', text)

    text = re.sub(r'[^a-zA-Z0-9,\s]', '', text)

    ingredients = text.split(',')

    ingredients = [i.strip() for i in ingredients]

    ingredients = list(set(ingredients))

    return " ".join(ingredients)

In [8]:
data['clean_ingreds'] = data['clean_ingreds'].apply(
    clean_ingredients
)

In [9]:
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=5000
)

tfidf_matrix = tfidf.fit_transform(
    data['clean_ingreds']
)

In [10]:
cosine_sim = cosine_similarity(tfidf_matrix)

In [11]:
svd = TruncatedSVD(
    n_components=50,
    random_state=42
)

svd_features = svd.fit_transform(tfidf_matrix)

tsne = TSNE(
    n_components=2,
    perplexity=30,
    n_iter=1000,
    random_state=42
)

tsne_features = tsne.fit_transform(
    svd_features
)

data['X'] = tsne_features[:, 0]
data['Y'] = tsne_features[:, 1]

In [12]:
unique_types = ['Moisturiser', 'Serum', 'Oil', 'Mist', 'Balm', 'Mask', 'Peel',
       'Eye Care', 'Cleanser', 'Toner', 'Exfoliator', 'Bath Salts',
       'Body Wash', 'Bath Oil']

source = ColumnDataSource(data)

plot = figure(title = "Mapped Similarities", width = 800, height = 600)
plot.xaxis.axis_label = "t-SNE 1"
plot.yaxis.axis_label = 't-SNE 2'

plot.circle(x = 'X', y = 'Y', source = source, fill_alpha=0.7, size=10,
           color = '#c0a5e3', alpha = 1)

plot.background_fill_color = "#E9E9E9"
plot.background_fill_alpha = 0.3

hover = HoverTool(tooltips=[('Product', '@product_name'), ('Price', '@price')])
plot.add_tools(hover)

def type_updater(product_type = unique_types[0]):
    new_data = {'X' : data[data['product_type'] == product_type]['X'],
                'Y' : data[data['product_type'] == product_type]['Y'],
                'product_name' : data[data['product_type'] == product_type]['product_name'],
                'price' : data[data['product_type'] == product_type]['price']}
    source.data = new_data
    push_notebook()

output_notebook()
show(plot, notebook_handle = True)
interact(type_updater, product_type = unique_types)

interactive(children=(Dropdown(description='product_type', options=('Moisturiser', 'Serum', 'Oil', 'Mist', 'Ba…

<function __main__.type_updater(product_type='Moisturiser')>

In [13]:
skin_concerns = {

    "acne": [
        "salicylic acid",
        "niacinamide",
        "benzoyl peroxide"
    ],

    "dryness": [
        "hyaluronic acid",
        "glycerin",
        "ceramide"
    ],

    "pigmentation": [
        "vitamin c",
        "kojic acid",
        "alpha arbutin"
    ],

    "sensitive_skin": [
        "centella",
        "allantoin",
        "panthenol"
    ]
}

In [14]:
avoid_sensitive_skin = [
    "fragrance",
    "alcohol denat",
    "essential oil"
]

In [15]:
def safety_penalty(ingredients):

    penalty = 0

    for item in avoid_sensitive_skin:

        if item in ingredients:
            penalty += 1

    return penalty

In [16]:
def concern_score(ingredients, concern):

    score = 0

    for ingredient in skin_concerns[concern]:

        if ingredient in ingredients:
            score += 1

    return score

In [17]:
def explain_recommendation(ingredients, concern):

    matched = []

    for ingredient in skin_concerns[concern]:

        if ingredient in ingredients:
            matched.append(ingredient)

    return matched

In [18]:
brand_list = ["111skin", "a'kin", "acorelle", "adam revolution", "aesop", "ahava", "alchimie forever",
             "algenist", "alpha-h", "ambre solaire", "ameliorate", "american crew", "anthony", "antipodes",
             "apivita", "argentum", "ark skincare", "armani", "aromatherapy associates", "aromaworks", "aromatica",
             "aurelia probiotic skincare", "aurelia skincare",
             "australian bodycare", "avant skincare", "aveda", "aveeno", "avene", "avène",
             "bakel", "balance me", "barber pro", "bareminerals", "barry m cosmetics",
             "baxter of california", "bbb london", "beautypro", "benefit", "benton", "bioderma",
             "bioeffect", "bloom & blossom", "bloom and blossom", "bobbi brown", "bondi sands", "bubble t", "bulldog", "burt's bees",
             "by terry", "carita", "caudalie", "cerave", "chantecaille", "clinique",
             "comfort zone", "connock london", "cosmetics 27", "cosrx", "cowshed", "crystal clear",
             "cult51", "darphin", "dear, klairs", "decleor", "decléor", "dermalogica", "dhc", "doctors formula",
             "dr. brandt", "dr brandt", "dr. hauschka", "dr hauschka", "dr. jackson's", "dr.jart+", "dr. lipp",
             "dr botanicals", "dr dennis", "dr. pawpaw", "ecooking", "egyptian magic",
             "eisenberg", "elemental herbology", "elemis", "elizabeth arden", "embryolisse",
             "emma hardie", "erno laszlo", "espa", "estée lauder", "estee lauder", "eucerin",
             "eve lom", "eve rebirth", "fade out", "farmacy", "filorga", "first aid beauty", "fit", "foreo",
             "frank body", "freezeframe", "gallinée", "garnier", "gatineau", "glamglow", "goldfaden md",
             "green people", "hawkins and brimble", "holika holika", "house 99", "huxley",
             "ilapothecary", "ila-spa", "indeed labs", "inika", "instant effects", "institut esthederm", "ioma", "klorane",
             "j.one", "jack black", "james read", "jason", "jo malone london", "juice beauty", "jurlique",
             "korres", "l:a bruket", "l'oréal men expert", "l'oreal men expert", "l'oréal paris", "l'oreal paris",
             "l’oréal paris", "lab series skincare for men",
             "lancaster", "lancer skincare", "lancôme", "lancome", "lanolips", "la roche-posay", "laura mercier",
             "liftlab", "little butterfly london", "lixirskin", "liz earle", "love boo",
             "löwengrip", "lowengrip", "lumene", "mac", "madara", "mádara", "magicstripes", "magnitone london",
             "mama mio", "mancave", "manuka doctor", "mauli", "mavala", "maybelline", "medik8", "men-u", "menaji", "molton brown", "moroccanoil",
             "monu", "murad", "naobay", "nars", "natio", "natura bissé", "natura bisse",
             "neal's yard remedies", "neom", "neostrata", "neutrogena", "niod", "nip+fab", "nuxe", "nyx",
             "oh k!", "omorovicza", "origins", "ortigia fico", "oskia", "ouai", "pai ", "paula's choice", "payot",
             "perricone md", "pestle & mortar", "pestle and mortar", "peter thomas roth",
             "philosophy", "pierre fabre", "pixi", "piz buin", "polaar", "prai", "project lip",
             "radical skincare", "rapideye", "rapidlash", "real chemistry", "recipe for men",
             "ren ", "renu", "revolution beauty", "revolution skincare", "rituals", "rmk", "rodial", "roger&gallet", "salcura",
             "sanctuary spa", "sanoflore", "sarah chapman", "sea magik", "sepai",
             "shaveworks", "shea moisture", "shiseido", "skin79", "skin authority", "skinceuticals",
             "skinchemists", "skindoctors", "skin doctors", "skinny tan", "sol de janeiro", "spa magik organiks",
              "st. tropez", "starskin", "strivectin", "sukin",
             "svr", "swiss clinic", "talika", "tan-luxe", "tanorganic", "tanworx", "thalgo", "the chemistry brand",
             "the hero project", "the inkey list", "the jojoba company", "the ordinary",
             "the organic pharmacy", "the ritual of namasté", "this works", "too faced", "trilogy", "triumph and disaster",
             "ultrasun", "uppercut deluxe", "urban decay", "uriage", "verso", "vichy",
             "vida glow", "vita liberata", "wahl", "weleda", "westlab", "wilma schumann", "yes to",
             "ysl", "zelens"]
brand_list = sorted(brand_list, key=len, reverse=True)

In [19]:
data['brand'] = data['product_name'].str.lower()
k=0
for i in data['brand']:
    for j in brand_list:
        if j in i:
            data['brand'][k] = data['brand'][k].replace(i, j.title())
    k+=1

data

,product_name,product_url,product_type,clean_ingreds,price,X,Y,brand
0,The Ordinary Natural Moisturising Factors + HA...,https://www.lookfantastic.com/the-ordinary-nat...,Moisturiser,oleic acid stearyl alcohol trehalose lecithin ...,£5.20,17.446857,45.700558,The Ordinary
1,CeraVe Facial Moisturising Lotion SPF 25 52ml,https://www.lookfantastic.com/cerave-facial-mo...,Moisturiser,homosalate behentrimonium methosulfate stearic...,£13.00,61.450993,1.554789,Cerave
2,The Ordinary Hyaluronic Acid 2% + B5 Hydration...,https://www.lookfantastic.com/the-ordinary-hya...,Moisturiser,caprylyl glycol phenoxyethanol sodium hyaluron...,£6.20,11.916991,39.483494,The Ordinary
3,AMELIORATE Transforming Body Lotion 200ml,https://www.lookfantastic.com/ameliorate-trans...,Moisturiser,limonene hydroxyethyl cellulose cetearyl gluco...,£22.50,21.459919,-8.563513,Ameliorate
4,CeraVe Moisturising Cream 454g,https://www.lookfantastic.com/cerave-moisturis...,Moisturiser,petrolatum behentrimonium methosulfate cetyl a...,£16.00,62.684772,-1.173676,Cerave
...,...,...,...,...,...,...,...,...
1133,Elemis Life Elixirs Embrace Bath and Shower El...,https://www.lookfantastic.com/elemis-life-elix...,Bath Oil,amaranthus caudatus seed extract limonene papa...,£55.00,-30.616154,-16.647388,Elemis
1134,Love Boo Splendidly Soothing Bath Soak (250ml),https://www.lookfantastic.com/love-boo-splendi...,Bath Oil,lauryl glucoside stearyl citrate pelargonium g...,£10.99,4.889411,-25.858852,Love Boo
1135,Elemis Life Elixirs Fortitude Bath and Shower ...,https://www.lookfantastic.com/elemis-life-elix...,Bath Oil,amaranthus caudatus seed extract limonene papa...,£55.00,-30.508547,-16.951105,Elemis
1136,Connock London Kukui Oil Soothing Bath & Showe...,https://www.lookfantastic.com/connock-london-k...,Bath Oil,aleurites moluccanus seed oil coumarin ppg15 s...,£36.00,-27.630379,-10.488802,Connock London


In [20]:
sorted(data.brand.unique())

["A'Kin",
 'Acorelle',
 'Aesop',
 'Ahava',
 'Alchimie Forever',
 'Alpha-H',
 'Ambre Solaire',
 'Ameliorate',
 'Antipodes',
 'Apivita',
 'Ark Skincare',
 'Armani',
 'Aromatherapy Associates',
 'Aromaworks',
 'Aurelia Probiotic Skincare',
 'Aurelia Skincare',
 'Australian Bodycare',
 'Avant Skincare',
 'Aveda',
 'Aveeno',
 'Avene',
 'Avène',
 'Balance Me',
 'Barber Pro',
 'Bareminerals',
 'Bbb London',
 'Beautypro',
 'Benefit',
 'Benton',
 'Bioderma',
 'Bloom & Blossom',
 'Bloom And Blossom',
 'Bobbi Brown',
 'Bondi Sands',
 'Bubble T',
 'Bulldog',
 "Burt'S Bees",
 'By Terry',
 'Caudalie',
 'Cerave',
 'Chantecaille',
 'Clinique',
 'Comfort Zone',
 'Connock London',
 'Cosrx',
 'Cowshed',
 'Crystal Clear',
 'Darphin',
 'Dear, Klairs',
 'Decléor',
 'Dermalogica',
 'Dhc',
 'Dr Brandt',
 'Dr Dennis',
 'Dr Hauschka',
 'Dr. Brandt',
 'Dr. Hauschka',
 'Dr. Pawpaw',
 'Dr.Jart+',
 'Egyptian Magic',
 'Elemental Herbology',
 'Elemis',
 'Elizabeth Arden',
 'Embryolisse',
 'Emma Hardie',
 'Erno Laszlo

In [21]:
data['brand'] = data['brand'].replace(['Aurelia Probiotic Skincare'],'Aurelia Skincare')
data['brand'] = data['brand'].replace(['Avene'],'Avène')
data['brand'] = data['brand'].replace(['Bloom And Blossom'],'Bloom & Blossom')
data['brand'] = data['brand'].replace(['Dr Brandt'],'Dr. Brandt')
data['brand'] = data['brand'].replace(['Dr Hauschka'],'Dr. Hauschka')
data['brand'] = data['brand'].replace(["L'oreal Paris", 'L’oréal Paris'], "L'oréal Paris")

In [22]:
def recommender(
    product_name,
    concern='acne',
    top_n=5
):

    idx = data[
        data['product_name'] == product_name
    ].index[0]

    product_type = data.iloc[idx]['product_type']

    brand_search = data.iloc[idx]['brand']

    similarity_scores = list(
        enumerate(cosine_sim[idx])
    )

    recommendations = []

    for i, sim_score in similarity_scores:

        product = data.iloc[i]
         # same product type only
        if product['product_type'] != product_type:
            continue

        # avoid same brand repetition
        if product['brand'] == brand_search:
            continue

        # avoid weak matches
        if sim_score < 0.2:
            continue

        ingredients = product['clean_ingreds']

        concern_match = concern_score(
            ingredients,
            concern
        )

        penalty = safety_penalty(
            ingredients
        )

        final_score = (
            0.7 * sim_score +
            0.3 * concern_match -
            0.1 * penalty
        )

        recommendations.append(
            (i, final_score)
        )

    recommendations = sorted(
        recommendations,
        key=lambda x: x[1],
        reverse=True
    )

    recommendations = recommendations[1:top_n+1]

    output = []

    for i, score in recommendations:

        product = data.iloc[i]

        explanation = explain_recommendation(
            product['clean_ingreds'],
            concern
        )

        output.append({

            'Product': product['product_name'],
            'Brand': product['brand'],
            'Product Type': product['product_type'],
            'Score': round(score, 2),
            'Why Recommended': ", ".join(explanation)
        })

    return pd.DataFrame(output)

In [23]:
recommender("Origins GinZing™ Energy-Boosting Tinted Moisturiser SPF40 50ml")

,Product,Brand,Product Type,Score,Why Recommended
0,Clinique Moisture Surge SPF25 Sheertint Hydrat...,Clinique,Moisturiser,0.33,
1,Estée Lauder NightWear Plus Anti-Oxidant Night...,Estée Lauder,Moisturiser,0.24,
2,Estée Lauder DayWear Advanced Multi-Protection...,Estée Lauder,Moisturiser,0.22,
3,Estée Lauder DayWear Multi-Protection Anti-Oxi...,Estée Lauder,Moisturiser,0.22,
4,Elemis Pro-Collagen Marine Cream SPF30 50ml,Elemis,Moisturiser,0.20,


In [24]:
recommender('Avène Antirougeurs Jour Redness Relief Moisturizing Protecting Cream (40ml)')

,Product,Brand,Product Type,Score,Why Recommended
0,Clinique Moisture Surge SPF25 Sheertint Hydrat...,Clinique,Moisturiser,0.17,
1,Origins GinZing™ Energy-Boosting Tinted Moistu...,Origins,Moisturiser,0.16,
2,Eucerin Anti-Pigment SPF30 Day Cream 50ml,Eucerin,Moisturiser,0.04,


In [25]:
recommender('Bondi Sands Everyday Liquid Gold Gradual Tanning Oil 270ml')

,Product,Brand,Product Type,Score,Why Recommended
0,Garnier Organic Lavandin Glow Oil 30ml,Garnier,Oil,0.17,
1,Mama Mio The Tummy Rub Oil 120ml,Mama Mio,Oil,0.16,
2,Mama Mio The Tummy Rub Oil 200ml,Mama Mio,Oil,0.16,
3,The Chemistry Brand Glow Oil 100ml,The Chemistry Brand,Oil,0.16,
4,TanOrganic Self-Tanning Oil - Brown (100ml),Tanorganic,Oil,0.09,


In [26]:
recommender('Sukin Rose Hip Oil (25ml)')

,Product,Brand,Product Type,Score,Why Recommended
0,Trilogy Certified Organic Rosehip Oil 45ml,Trilogy,Oil,0.70,
1,Pai Skincare Rosehip BioRegenerate Oil 30ml,Pai,Oil,0.51,
2,Natio Ageless Rosehip Oil Cold Pressed 15ml,Natio,Oil,0.42,
3,PIXI Rose Oil Blend 30ml,Pixi,Oil,0.33,
4,Sanctuary Spa 10-in-1 Super Secret Facial Oil ...,Sanctuary Spa,Oil,0.33,


In [27]:
recommender('La Roche-Posay Anthelios Anti-Shine Sun Protection Invisible SPF50+ Face Mist 75ml')

,Product,Brand,Product Type,Score,Why Recommended
0,Garnier Ambre Solaire Sensitive Hypoallergenic...,Ambre Solaire,Mist,0.46,
1,Garnier Ambre Solaire Dry Mist Fast Absorbing ...,Ambre Solaire,Mist,0.45,


In [28]:
recommender('Clinique Even Better Clinical Radical Dark Spot Corrector + Interrupter 30ml')

,Product,Brand,Product Type,Score,Why Recommended
0,Estée Lauder Perfectionist Pro Rapid Brighteni...,Estée Lauder,Serum,0.53,salicylic acid
1,Elizabeth Arden Prevage Advanced Daily Serum,Elizabeth Arden,Serum,0.49,salicylic acid
2,Aveda Hand Relief Night Renewal Serum 30ml,Aveda,Serum,0.44,salicylic acid
3,NIP+FAB Salicylic Fix Serum Extreme 2% 50ml,Nip+Fab,Serum,0.44,salicylic acid
4,Estée Lauder Idealist Pore Minimizing Skin Ref...,Estée Lauder,Serum,0.32,


In [29]:
recommender('Garnier Organic Argan Mist 150ml')

,Product,Brand,Product Type,Score,Why Recommended
0,Salcura Antiac Acne Clearing Spray (100ML),Salcura,Mist,0.27,
1,Zelens PROVITAMIN D Fortifying Facial Mist 50ml,Zelens,Mist,0.21,
2,The Ritual of Namasté Urban Hydrating Mist 100ml,The Ritual Of Namasté,Mist,0.17,
3,Acorelle Pure Harvest Body Perfume - 100ml,Acorelle,Mist,0.16,


In [30]:
recommender('Shea Moisture 100% Virgin Coconut Oil Daily Hydration Body Wash 384ml')

,Product,Brand,Product Type,Score,Why Recommended
0,Aveeno Daily Moisturising Body Wash 300ml,Aveeno,Body Wash,0.19,
1,Pai Skincare Gentle Genius Camellia and Bergam...,Pai,Body Wash,0.18,
2,APIVITA Pure Jasmine Mini Shower Gel with Esse...,Apivita,Body Wash,0.17,
3,Elemental Herbology Neroli and Rose Damask Bod...,Elemental Herbology,Body Wash,0.16,
4,Caudalie Thé des Vignes Shower Gel 200ml,Caudalie,Body Wash,0.16,


In [31]:
recommender('JASON Soothing Aloe Vera Body Wash 887ml')

,Product,Brand,Product Type,Score,Why Recommended
0,DECLÉOR Luxury Size Lavender Shower Gel 400ml,Decléor,Body Wash,0.50,salicylic acid
1,L'Oréal Paris Men Expert Hydra Power Shower Ge...,L'Oréal Paris,Body Wash,0.50,salicylic acid
2,DECLÉOR Luxury Size Rose Shower Gel 400ml,Decléor,Body Wash,0.49,salicylic acid
3,DECLÉOR Luxury Size Neroli Shower Gel 400ml,Decléor,Body Wash,0.49,salicylic acid
4,L'Oréal Paris Men Expert Clean Power Shower Ge...,L'Oréal Paris,Body Wash,0.49,salicylic acid


In [32]:
def skincare_assistant(
    concern='acne',
    product_type='Serum',
    avoid_ingredient='fragrance'
):

    filtered_data = data[
        data['product_type'] == product_type
    ]

    recommendations = []

    for idx in filtered_data.index:

        product = filtered_data.loc[idx]

        ingredients = product['clean_ingreds']

        # skip avoided ingredients
        if avoid_ingredient.lower() in ingredients:
            continue

        concern_match = concern_score(
            ingredients,
            concern
        )

        penalty = safety_penalty(
            ingredients
        )

        final_score = (
            concern_match -
            0.2 * penalty
        )

        recommendations.append({

            'Product': product['product_name'],
            'Brand': product['brand'],
            'Score': final_score,
            'Why Recommended':
            ", ".join(
                explain_recommendation(
                    ingredients,
                    concern
                )
            )
        })

    recommendations = sorted(
        recommendations,
        key=lambda x: x['Score'],
        reverse=True
    )

    return pd.DataFrame(
        recommendations[:5]
    )

In [33]:
interact(
    skincare_assistant,

    concern=[
        'acne',
        'dryness',
        'pigmentation',
        'sensitive_skin'
    ],

    product_type=sorted(
        data['product_type'].unique()
    ),

    avoid_ingredient=[
        'fragrance',
        'alcohol',
        'essential oil'
    ]
)

interactive(children=(Dropdown(description='concern', options=('acne', 'dryness', 'pigmentation', 'sensitive_s…

<function __main__.skincare_assistant(concern='acne', product_type='Serum', avoid_ingredient='fragrance')>


The content-based recommendation engine was successfully developed using cosine similarity. The recommendation engine enables users to make better decisions on which product to purchase, as many recommendations contain products that are better value for money. It also has the potential to improve business for less popular brands by recommending their products.

